In [11]:
import pandas as pd

In [2]:
food = pd.read_csv("/home/jithesh/Desktop/food_suitability_bot/data/raw/food.csv")
food_nutrient = pd.read_csv("/home/jithesh/Desktop/food_suitability_bot/data/raw/food_nutrient.csv")

In [3]:
# Selected Nutrient IDs (from USDA SR Legacy)
NUTRIENT_MAP = {
    1008: 'calories',
    1003: 'protein_g',
    1004: 'fat_g',
    1005: 'carbs_g',
    2000: 'sugar_g',
    1079: 'fiber_g',
    1093: 'sodium_mg',
}

In [4]:
# Copying Selected Nutrients from food_nutrient

filtered = food_nutrient[food_nutrient['nutrient_id'].isin(NUTRIENT_MAP.keys())].copy() 
filtered['nutrient_id'].unique()

array([1003, 1079, 1093, 1004, 1005, 1008, 2000])

In [5]:
# Mapping NUTRIENT_MAP to filtered 
filtered['nutrient_name'] = filtered['nutrient_id'].map(NUTRIENT_MAP)
filtered.head(1)

,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired,nutrient_name
0,1283674,167512,1003,5.88,1,46.0,NaN,NaN,NaN,NaN,NaN,protein_g


In [6]:
# Pivot so each food is one row with nutrient columns

pivoted = filtered.pivot_table(
    index='fdc_id',
    columns='nutrient_name',
    values='amount',
    aggfunc='first'
).reset_index()

pivoted.head()

nutrient_name,fdc_id,calories,carbs_g,fat_g,fiber_g,protein_g,sodium_mg,sugar_g
0,167512,307.0,41.18,13.24,1.2,5.88,1059.0,5.88
1,167513,330.0,53.42,11.27,1.4,4.34,780.0,21.34
2,167514,377.0,79.80,3.70,NaN,6.10,2182.0,NaN
3,167515,232.0,46.00,1.80,NaN,8.00,345.0,NaN
4,167516,273.0,41.05,9.22,2.2,6.58,621.0,4.30


In [7]:
# Merge with food names
merged = pivoted.merge(food[['fdc_id', 'description']], on='fdc_id')
merged = merged.rename(columns={'description': 'food_name'})
merged.head()

,fdc_id,calories,carbs_g,fat_g,fiber_g,protein_g,sodium_mg,sugar_g,food_name
0,167512,307.0,41.18,13.24,1.2,5.88,1059.0,5.88,"Pillsbury Golden Layer Buttermilk Biscuits, Ar..."
1,167513,330.0,53.42,11.27,1.4,4.34,780.0,21.34,"Pillsbury, Cinnamon Rolls with Icing, refriger..."
2,167514,377.0,79.80,3.70,NaN,6.10,2182.0,NaN,"Kraft Foods, Shake N Bake Original Recipe, Coa..."
3,167515,232.0,46.00,1.80,NaN,8.00,345.0,NaN,"George Weston Bakeries, Thomas English Muffins"
4,167516,273.0,41.05,9.22,2.2,6.58,621.0,4.30,"Waffles, buttermilk, frozen, ready-to-heat"


In [8]:
# Drop - rows with too many missing values
merged = merged.dropna(thresh=5)

# Fill remaining NaN with 0
merged = merged.fillna(0)

# Clean food names to lowercase
merged['food_name'] = merged['food_name'].str.lower().str.strip()

In [9]:
# Keep only relevant columns in clean order
cols = ['fdc_id', 'food_name', 'calories', 'protein_g', 'fat_g',
        'carbs_g', 'sugar_g', 'fiber_g', 'sodium_mg']
# Only keep cols that exist
cols = [c for c in cols if c in merged.columns]
merged = merged[cols]
merged.head()

,fdc_id,food_name,calories,protein_g,fat_g,carbs_g,sugar_g,fiber_g,sodium_mg
0,167512,"pillsbury golden layer buttermilk biscuits, ar...",307.0,5.88,13.24,41.18,5.88,1.2,1059.0
1,167513,"pillsbury, cinnamon rolls with icing, refriger...",330.0,4.34,11.27,53.42,21.34,1.4,780.0
2,167514,"kraft foods, shake n bake original recipe, coa...",377.0,6.10,3.70,79.80,0.00,0.0,2182.0
3,167515,"george weston bakeries, thomas english muffins",232.0,8.00,1.80,46.00,0.00,0.0,345.0
4,167516,"waffles, buttermilk, frozen, ready-to-heat",273.0,6.58,9.22,41.05,4.30,2.2,621.0


In [ ]:
print(f"Final shape: {merged.shape}")

Final shape: (7793, 9)
